# 🔍 Verificar Documentos Faltantes en Supabase

Este notebook compara los archivos HTML en Google Drive con los documentos en Supabase y muestra qué falta cargar.

## 1. Instalación de Dependencias

In [ ]:
!pip install supabase google-api-python-client google-auth-httplib2 google-auth-oauthlib pandas -q

## 2. Configuración

In [ ]:
import os
from google.colab import userdata
from supabase import create_client
from google.oauth2 import service_account
from googleapiclient.discovery import build
import pandas as pd

# Configuración de Supabase
SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_KEY = userdata.get('SUPABASE_ANON_KEY')
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Configuración de Google Drive (usa tu service account JSON)
# Sube tu archivo credentials.json a Colab
from google.colab import files
print("📤 Sube tu archivo de credenciales de Google (service account JSON):")
uploaded = files.upload()

CREDENTIALS_FILE = list(uploaded.keys())[0]
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

credentials = service_account.Credentials.from_service_account_file(
    CREDENTIALS_FILE, scopes=SCOPES
)
drive_service = build('drive', 'v3', credentials=credentials)

print("✅ Configuración completada")

## 3. Definir Carpetas de Drive

In [ ]:
# REEMPLAZA CON TUS IDs REALES DE GOOGLE DRIVE
FOLDERS = {
    'Sala_Constitucional': {
        'id': 'TU_ID_AQUI',
        'tipo': 'sentencia',
        'sala': 'constitucional'
    },
    'Sala_Penal': {
        'id': 'TU_ID_AQUI',
        'tipo': 'sentencia',
        'sala': 'penal'
    },
    'Sustanciacion_Constitucional': {
        'id': 'TU_ID_AQUI',
        'tipo': 'sentencia',
        'sala': 'sustanciacion'
    },
    'Sala_Penal_Juris': {
        'id': 'TU_ID_AQUI',
        'tipo': 'jurisprudencia',
        'sala': 'penal'
    },
    'Sala_Constitucional_Juris': {
        'id': 'TU_ID_AQUI',
        'tipo': 'jurisprudencia',
        'sala': 'constitucional'
    },
    'Resoluciones_Constitucional': {
        'id': 'TU_ID_AQUI',
        'tipo': 'resolucion',
        'sala': 'constitucional'
    },
    # Agrega más carpetas aquí si las tienes
}

print(f"📁 Carpetas configuradas: {len(FOLDERS)}")

## 4. Obtener Documentos de Supabase

In [ ]:
print("📊 Obteniendo documentos de Supabase...")

# Obtener todos los IDs de documentos en Supabase
supabase_docs = []
offset = 0
batch_size = 1000

while True:
    response = supabase.table('documentos').select('id, tipo, sala').range(offset, offset + batch_size - 1).execute()
    if not response.data:
        break
    supabase_docs.extend(response.data)
    offset += batch_size
    print(f"  Descargados: {len(supabase_docs)}...", end='\r')

df_supabase = pd.DataFrame(supabase_docs)
print(f"\n✅ Total en Supabase: {len(df_supabase):,}")

# Convertir IDs a set para búsqueda rápida
supabase_ids = set(df_supabase['id'].tolist())

# Estadísticas por tipo y sala
print("\n📊 Distribución en Supabase:")
print(df_supabase.groupby(['tipo', 'sala']).size().sort_values(ascending=False))

## 5. Escanear Drive y Comparar

In [ ]:
def listar_archivos_drive(folder_id, tipo, sala):
    """Lista todos los archivos HTML en una carpeta de Drive"""
    archivos = []
    page_token = None
    
    query = f"'{folder_id}' in parents and mimeType='text/html' and trashed=false"
    
    while True:
        response = drive_service.files().list(
            q=query,
            fields='nextPageToken, files(id, name)',
            pageToken=page_token,
            pageSize=1000
        ).execute()
        
        for file in response.get('files', []):
            archivos.append({
                'id': file['id'],
                'nombre': file['name'],
                'tipo': tipo,
                'sala': sala
            })
        
        page_token = response.get('nextPageToken')
        if not page_token:
            break
    
    return archivos

print("🔍 Escaneando Google Drive...\n")

drive_todos = []
faltantes_por_carpeta = {}

for nombre, config in FOLDERS.items():
    print(f"📂 {nombre}...")
    
    archivos = listar_archivos_drive(config['id'], config['tipo'], config['sala'])
    drive_todos.extend(archivos)
    
    # Identificar faltantes
    ids_drive = set([a['id'] for a in archivos])
    faltantes = ids_drive - supabase_ids
    
    faltantes_por_carpeta[nombre] = {
        'total_drive': len(archivos),
        'faltantes': len(faltantes),
        'ids_faltantes': list(faltantes)
    }
    
    print(f"   Total: {len(archivos):,} | Faltantes: {len(faltantes):,}")

print(f"\n✅ Total en Drive: {len(drive_todos):,}")

## 6. Resumen de Faltantes

In [ ]:
print("\n" + "="*80)
print("📊 RESUMEN COMPLETO")
print("="*80)

total_drive = len(drive_todos)
total_supabase = len(df_supabase)
total_faltantes = sum([f['faltantes'] for f in faltantes_por_carpeta.values()])

print(f"\n📁 Total archivos en Drive:    {total_drive:>10,}")
print(f"📊 Total docs en Supabase:     {total_supabase:>10,}")
print(f"❌ Total faltantes:            {total_faltantes:>10,}")
print(f"✅ Porcentaje cargado:         {(total_supabase/total_drive*100):>9.2f}%")

print("\n" + "="*80)
print("📋 DESGLOSE POR CARPETA")
print("="*80)

df_resumen = pd.DataFrame([
    {
        'Carpeta': nombre,
        'En Drive': info['total_drive'],
        'Faltantes': info['faltantes'],
        '% Completo': f"{((info['total_drive']-info['faltantes'])/info['total_drive']*100):.1f}%"
    }
    for nombre, info in faltantes_por_carpeta.items()
])

df_resumen = df_resumen.sort_values('Faltantes', ascending=False)
print(df_resumen.to_string(index=False))

print("\n" + "="*80)

## 7. Exportar Lista de Faltantes

In [ ]:
# Crear DataFrame con todos los faltantes
faltantes_detalle = []

for nombre, info in faltantes_por_carpeta.items():
    if info['faltantes'] > 0:
        for doc_id in info['ids_faltantes']:
            # Buscar info del documento
            doc = next((d for d in drive_todos if d['id'] == doc_id), None)
            if doc:
                faltantes_detalle.append({
                    'carpeta': nombre,
                    'id': doc['id'],
                    'nombre': doc['nombre'],
                    'tipo': doc['tipo'],
                    'sala': doc['sala']
                })

df_faltantes = pd.DataFrame(faltantes_detalle)

if len(df_faltantes) > 0:
    print(f"\n💾 Exportando {len(df_faltantes):,} documentos faltantes...")
    df_faltantes.to_csv('documentos_faltantes.csv', index=False)
    print("✅ Archivo guardado: documentos_faltantes.csv")
    
    # Descargar el archivo
    files.download('documentos_faltantes.csv')
    
    print("\n📋 Primeros 10 documentos faltantes:")
    print(df_faltantes.head(10).to_string(index=False))
else:
    print("\n✅ ¡No hay documentos faltantes! Todo está cargado.")

## 8. Análisis Adicional (Opcional)

In [ ]:
# Verificar si hay documentos en Supabase que NO están en Drive
drive_ids = set([d['id'] for d in drive_todos])
extras_en_supabase = supabase_ids - drive_ids

print(f"\n🔍 ANÁLISIS ADICIONAL:")
print(f"\nDocumentos en Supabase que NO están en Drive: {len(extras_en_supabase):,}")

if len(extras_en_supabase) > 0:
    print("\n⚠️  Esto puede significar:")
    print("  1. Documentos eliminados de Drive pero aún en Supabase")
    print("  2. Documentos de otras fuentes")
    print("  3. Errores en la comparación")
    
    # Mostrar algunos ejemplos
    print("\n📋 Primeros 10 IDs extras:")
    for i, doc_id in enumerate(list(extras_en_supabase)[:10], 1):
        print(f"  {i}. {doc_id}")